In [1]:
# ── New, separate Colab notebook — does not touch your running sweep ──
!pip install -q git+https://github.com/Blealtan/efficient-kan.git

from google.colab import drive
drive.mount("/content/drive")

import sys
sys.path.insert(0, "/content/drive/MyDrive/Thesis/Code")

import torch
import pandas as pd
from sparse_kan import SparseKAN

# Small synthetic taxonomy, same shape as your earlier sanity checks --
# 2 themes so there's a genuine masked/unmasked contrast, and a subtheme
# with more than 1 active feature so "fan-in" is meaningfully > 1.
fake_tax = pd.DataFrame({
    "column":        ["f1", "f2", "f3", "f4", "f5"],
    "subtheme_id":   ["01_01", "01_01", "01_01", "02_01", "02_01"],
    "subtheme_name": ["SubA", "SubA", "SubA", "SubB", "SubB"],
    "theme_id":      ["01", "01", "01", "02", "02"],
    "theme_name":    ["ThemeX", "ThemeX", "ThemeX", "ThemeY", "ThemeY"],
})
fcols = ["f1", "f2", "f3", "f4", "f5"]

model = SparseKAN.from_taxonomy(fake_tax, fcols, grid_size=5, spline_order=3,
                                grid_range=[-1, 1])

# SubA (subtheme index 0) has fan-in 3 -- real bound should be 1/sqrt(3) ≈ 0.577
mask_row = model.layer0.mask[0]          # which of f1..f5 feed subtheme 0
active_idx = mask_row.bool()
fan_in = int(mask_row.sum().item())
expected_bound = 1.0 / (fan_in ** 0.5)

actual_max = model.layer0.base_weight.data[0][active_idx].abs().max().item()

print(f"Fan-in for this subtheme: {fan_in}")
print(f"Expected bound (1/sqrt(fan_in)): {expected_bound:.4f}")
print(f"Actual max |weight| on active edges: {actual_max:.4f}")
print()
if actual_max < expected_bound * 0.3:
    print("BUG CONFIRMED: weights are far smaller than 1/sqrt(fan_in) --")
    print("the advanced-indexing .uniform_() call is not writing through.")
else:
    print("Weights match expected scale -- Bug A does NOT apply to this file.")

Mounted at /content/drive
Fan-in for this subtheme: 3
Expected bound (1/sqrt(fan_in)): 0.5774
Actual max |weight| on active edges: 0.3203

Weights match expected scale -- Bug A does NOT apply to this file.


In [2]:
from data_utils import load_split, load_theme_assignment
from pathlib import Path

THEMES_DIR = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/05_themes")
SPLITS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/04_splits")

tax = load_theme_assignment("agg_full_moments", THEMES_DIR)
data = load_split("Split_A", "agg_full_moments", SPLITS_DIR)

model = SparseKAN.from_taxonomy(tax, data["feature_cols"], grid_size=14,
                                spline_order=3, grid_range=[-5.5, 5.5])

import torch
mask = model.layer0.mask
n_check = 0
n_ok = 0
for q in range(mask.shape[0]):
    fan_in = int(mask[q].sum().item())
    if fan_in == 0:
        continue
    expected_bound = 1.0 / (fan_in ** 0.5)
    actual_max = model.layer0.base_weight.data[q][mask[q].bool()].abs().max().item()
    n_check += 1
    # actual_max should be <= expected_bound (it's a uniform draw within that range)
    # and should be in a plausible range for that many samples, not orders of magnitude off
    if actual_max <= expected_bound * 1.05:  # small tolerance
        n_ok += 1

print(f"Checked {n_check} subthemes with fan-in > 0")
print(f"{n_ok}/{n_check} have max weight within the correct fan-in-scaled bound")

Checked 331 subthemes with fan-in > 0
331/331 have max weight within the correct fan-in-scaled bound


In [3]:
import torch

# Quick check: does post-BN std actually match the sigma/sqrt(eps) prediction
# across a range of pre-BN std values, confirming eps dominates below ~0.316?

eps = 0.1
for sigma in [0.03, 0.06, 0.15, 0.35, 0.8, 1.0]:
    x = torch.randn(2048, 50) * sigma  # simulate a layer's raw pre-BN output
    bn = torch.nn.BatchNorm1d(50, affine=False, eps=eps)
    bn.train()
    y = bn(x)
    predicted = sigma / (sigma**2 + eps) ** 0.5
    print(f"sigma={sigma:.2f}  predicted_post_std={predicted:.3f}  "
          f"actual_post_std={y.std().item():.3f}")

sigma=0.03  predicted_post_std=0.094  actual_post_std=0.094
sigma=0.06  predicted_post_std=0.186  actual_post_std=0.186
sigma=0.15  predicted_post_std=0.429  actual_post_std=0.429
sigma=0.35  predicted_post_std=0.742  actual_post_std=0.742
sigma=0.80  predicted_post_std=0.930  actual_post_std=0.930
sigma=1.00  predicted_post_std=0.953  actual_post_std=0.953


In [4]:
import torch
import numpy as np
from data_utils import load_split

data = load_split("Split_A", "agg_full_moments", SPLITS_DIR)
x0 = torch.tensor(data["X_train"][:2048], dtype=torch.float32)

with torch.no_grad():
    x1 = model.layer0(x0)   # subtheme outputs, pre-BN

stds = x1.std(dim=0)
fan_in = model.layer0.mask.sum(dim=1)

# Correlation between fan-in and pre-BN std -- tests hypothesis 1 directly
corr = np.corrcoef(fan_in.numpy(), stds.numpy())[0, 1]
print(f"Correlation between fan-in and pre-BN std: {corr:.3f}")

# Which subthemes actually have the lowest std -- look up their names
low_std_idx = torch.argsort(stds)[:10]
for i in low_std_idx:
    print(f"  {model._subtheme_names[i]:<45} std={stds[i].item():.4f} "
          f"fan_in={int(fan_in[i].item())}")

Correlation between fan-in and pre-BN std: 0.539
  10_07 Institutional Trade Share Spread        std=0.0002 fan_in=3
  10_05 Institutional Trade Share               std=0.0008 fan_in=3
  06_20 Product Market Concentration Std        std=0.0011 fan_in=2
  06_17 Product Market Concentration            std=0.0020 fan_in=2
  13_28 Realised Tail Move                      std=0.0029 fan_in=2
  07_24 Order Book Pressure Imbalance Spread    std=0.0029 fan_in=3
  13_30 Realised Tail Move Spread               std=0.0036 fan_in=2
  05_07 Nominal Yield Level                     std=0.0039 fan_in=11
  05_03 Inflation Compensation Level            std=0.0039 fan_in=2
  07_08 Depth Deterioration Spread              std=0.0039 fan_in=2


In [6]:
import torch
import numpy as np
from data_utils import load_split, load_theme_assignment
from pathlib import Path
from sparse_kan import SparseKAN

THEMES_DIR = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/05_themes")
SPLITS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/04_splits")

tax = load_theme_assignment("agg_full_moments", THEMES_DIR)
data = load_split("Split_A", "agg_full_moments", SPLITS_DIR)
feature_cols = data["feature_cols"]          # <-- this was missing before

model = SparseKAN.from_taxonomy(tax, feature_cols, grid_size=14,
                                spline_order=3, grid_range=[-5.5, 5.5])

x0 = torch.tensor(data["X_train"][:2048], dtype=torch.float32)

# ── Re-confirm the weight-scale check (same as before, sanity re-run) ──
r = int(model.layer0.mask.sum(1).argmax())
active_r = model.layer0.mask[r].bool()
fan_in_r = int(model.layer0.mask[r].sum().item())
print(f"High-fan-in row check: fan_in={fan_in_r}, "
      f"expected_bound={1/fan_in_r**0.5:.4f}, "
      f"actual_max={model.layer0.base_weight.data[r][active_r].abs().max().item():.4f}")

# ── NEW: test the correlation-cancellation hypothesis on the worst offenders ──
with torch.no_grad():
    x1 = model.layer0(x0)

stds = x1.std(dim=0)
low_std_idx = torch.argsort(stds)[:5]

for q in low_std_idx:
    active_feats = model.layer0.mask[q].bool()
    feat_idx = active_feats.nonzero(as_tuple=True)[0]
    raw_inputs = x0[:, feat_idx]                  # raw feature values feeding this subtheme

    # correlation among this subtheme's own active raw features
    if raw_inputs.shape[1] > 1:
        corr_matrix = torch.corrcoef(raw_inputs.T)
        mean_abs_corr = corr_matrix[~torch.eye(len(feat_idx), dtype=bool)].abs().mean().item()
    else:
        mean_abs_corr = float('nan')

    weights = model.layer0.base_weight.data[q][active_feats]
    signs = torch.sign(weights)

    print(f"\n  {model._subtheme_names[q]}: pre-BN std={stds[q].item():.5f}, "
          f"fan_in={len(feat_idx)}")
    print(f"    mean |correlation| among active inputs: {mean_abs_corr:.3f}")
    print(f"    weight signs: {signs.tolist()}")

High-fan-in row check: fan_in=34, expected_bound=0.1715, actual_max=0.0241

  10_07 Institutional Trade Share Spread: pre-BN std=0.00034, fan_in=3
    mean |correlation| among active inputs: 0.993
    weight signs: [-1.0, 1.0, -1.0]

  05_05 Money Market Carry Return: pre-BN std=0.00079, fan_in=2
    mean |correlation| among active inputs: 0.914
    weight signs: [1.0, 1.0]

  06_20 Product Market Concentration Std: pre-BN std=0.00167, fan_in=2
    mean |correlation| among active inputs: 0.785
    weight signs: [-1.0, 1.0]

  05_03 Inflation Compensation Level: pre-BN std=0.00177, fan_in=2
    mean |correlation| among active inputs: 0.968
    weight signs: [1.0, -1.0]

  13_08 Index Implied Volatility Level: pre-BN std=0.00224, fan_in=5
    mean |correlation| among active inputs: 0.976
    weight signs: [-1.0, -1.0, 1.0, 1.0, -1.0]


In [7]:
ratios = []
for q in range(model.layer0.mask.shape[0]):
    active = model.layer0.mask[q].bool()
    fan_in = int(model.layer0.mask[q].sum().item())
    if fan_in < 5:  # skip tiny fan-in rows where sampling noise dominates
        continue
    expected = 1.0 / fan_in**0.5
    actual = model.layer0.base_weight.data[q][active].abs().max().item()
    ratios.append(actual / expected)

import numpy as np
ratios = np.array(ratios)
print(f"n={len(ratios)}, mean ratio={ratios.mean():.3f}, "
      f"median={np.median(ratios):.3f}, std={ratios.std():.3f}")
print(f"Ratios: {sorted(ratios)[:10]} ... {sorted(ratios)[-10:]}")

n=145, mean ratio=0.059, median=0.056, std=0.016
Ratios: [np.float64(0.028969174179208357), np.float64(0.03054857599633936), np.float64(0.0313689166567708), np.float64(0.03186337934836622), np.float64(0.031959276417842665), np.float64(0.034319445602139065), np.float64(0.03452868021813189), np.float64(0.03496461006576165), np.float64(0.03584183189359282), np.float64(0.037886116817599624)] ... [np.float64(0.08525040377010303), np.float64(0.08572217006961493), np.float64(0.08829518893159337), np.float64(0.08839495828766233), np.float64(0.0901440096791943), np.float64(0.09021207961347417), np.float64(0.09664969146251678), np.float64(0.0986638676747938), np.float64(0.11379767892999858), np.float64(0.140724360395859)]


In [8]:
import torch
import numpy as np
from data_utils import load_split, load_theme_assignment
from pathlib import Path
from sparse_mlp import SparseMLP

THEMES_DIR = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/05_themes")
SPLITS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/04_splits")

tax = load_theme_assignment("agg_full_moments", THEMES_DIR)
data = load_split("Split_A", "agg_full_moments", SPLITS_DIR)
feature_cols = data["feature_cols"]

model = SparseMLP.from_taxonomy(tax, feature_cols)
x0 = torch.tensor(data["X_train"][:2048], dtype=torch.float32)

# ── Check 1: ratio test (same as the KAN check) ──
ratios = []
for q in range(model.layer0.mask.shape[0]):
    active = model.layer0.mask[q].bool()
    fan_in = int(model.layer0.mask[q].sum().item())
    if fan_in < 5:
        continue
    expected = 1.0 / fan_in**0.5
    actual = model.layer0.weight.data[q][active].abs().max().item()
    ratios.append(actual / expected)

ratios = np.array(ratios)
print("=" * 70)
print("CHECK 1 -- ratio test (weight scale)")
print("=" * 70)
print(f"n={len(ratios)}, mean={ratios.mean():.4f}, median={np.median(ratios):.4f}, "
      f"std={ratios.std():.4f}")
print("Prediction: mean ~0.059, median ~0.056 (matching the KAN's numbers, "
      "since both inherit 1/sqrt(1699))")

# ── Check 2: high-fan-in row, direct comparison ──
r = int(model.layer0.mask.sum(1).argmax())
active_r = model.layer0.mask[r].bool()
fan_in_r = int(model.layer0.mask[r].sum().item())
actual_r = model.layer0.weight.data[r][active_r].abs().max().item()
print(f"\nCHECK 2 -- high fan-in row: fan_in={fan_in_r}, "
      f"expected={1/fan_in_r**0.5:.4f}, actual={actual_r:.4f}")

# ── Check 3: does this bug push SiLU into its near-linear regime? ──
with torch.no_grad():
    pre_silu_0 = model.layer0(x0)          # layer 0 output, BEFORE SiLU

pre_silu_std = pre_silu_0.std(dim=0)
print("\n" + "=" * 70)
print("CHECK 3 -- SiLU operating regime (layer 0)")
print("=" * 70)
print(f"pre-SiLU std across subthemes: min={pre_silu_std.min():.4f}  "
      f"median={pre_silu_std.median():.4f}  max={pre_silu_std.max():.4f}")

# SiLU(x) = x*sigmoid(x) ~= 0.5x + 0.25x^2 near x=0 (first-order Taylor)
# ratio of quadratic to linear term, evaluated at 1 std, estimates how
# "linear" the activation is behaving for a typical input to this unit
median_std = pre_silu_std.median().item()
quad_to_linear_ratio = 0.5 * median_std
print(f"\nEstimated quadratic/linear term ratio at median std: "
      f"{quad_to_linear_ratio:.4f}")
print("  (a few percent -> SiLU is operating in its near-linear regime; "
      "confirms the compounding effect predicted above)")
print("  (closer to 0.2-0.5 -> SiLU is genuinely bending, prediction not "
      "confirmed for this layer)")

# ── Check 4: does the SECOND layer show the same pattern, post-SiLU input? ──
with torch.no_grad():
    post_silu_0 = model.activation(pre_silu_0)   # what layer 1 actually receives
    pre_silu_1  = model.layer1(post_silu_0)

pre_silu_1_std = pre_silu_1.std(dim=0)
print("\n" + "=" * 70)
print("CHECK 4 -- SiLU operating regime (layer 1)")
print("=" * 70)
print(f"pre-SiLU std across themes: min={pre_silu_1_std.min():.4f}  "
      f"median={pre_silu_1_std.median():.4f}  max={pre_silu_1_std.max():.4f}")
median_std_1 = pre_silu_1_std.median().item()
print(f"Estimated quadratic/linear ratio at median std: "
      f"{0.5*median_std_1:.4f}")

CHECK 1 -- ratio test (weight scale)
n=145, mean=0.0578, median=0.0552, std=0.0175
Prediction: mean ~0.059, median ~0.056 (matching the KAN's numbers, since both inherit 1/sqrt(1699))

CHECK 2 -- high fan-in row: fan_in=34, expected=0.1715, actual=0.0242

CHECK 3 -- SiLU operating regime (layer 0)
pre-SiLU std across subthemes: min=0.0002  median=0.0274  max=0.1157

Estimated quadratic/linear term ratio at median std: 0.0137
  (a few percent -> SiLU is operating in its near-linear regime; confirms the compounding effect predicted above)
  (closer to 0.2-0.5 -> SiLU is genuinely bending, prediction not confirmed for this layer)

CHECK 4 -- SiLU operating regime (layer 1)
pre-SiLU std across themes: min=0.0006  median=0.0029  max=0.0053
Estimated quadratic/linear ratio at median std: 0.0014


# New set of tests after fixes

In [2]:
# ── New, separate Colab notebook — does not touch your running sweep ──
!pip install -q git+https://github.com/Blealtan/efficient-kan.git

from google.colab import drive
drive.mount("/content/drive")

import sys
sys.path.insert(0, "/content/drive/MyDrive/Thesis/Code")


import torch
import numpy as np
from data_utils import load_split, load_theme_assignment
from pathlib import Path
from sparse_kan import SparseKAN
from sparse_mlp import SparseMLP

THEMES_DIR = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/05_themes")
SPLITS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/04_splits")

def verify_init(model, dataset_name, model_name):
    ratios = []
    fan_ins = []
    for q in range(model.layer0.mask.shape[0]):
        active = model.layer0.mask[q].bool()
        fan_in = int(model.layer0.mask[q].sum().item())
        if fan_in < 5:
            continue
        expected = 1.0 / fan_in**0.5
        w = model.layer0.weight.data if hasattr(model.layer0, "weight") else model.layer0.base_weight.data
        actual = w[q][active].abs().max().item()
        ratios.append(actual / expected)
        fan_ins.append(fan_in)

    ratios = np.array(ratios)
    fan_ins = np.array(fan_ins)
    corr = np.corrcoef(fan_ins, ratios)[0, 1]

    print(f"  [{model_name} / {dataset_name}]  n={len(ratios)}  "
          f"mean_ratio={ratios.mean():.3f}  median={np.median(ratios):.3f}  "
          f"corr(fan_in, ratio)={corr:.3f}")
    print(f"    -> expect mean/median near 1.0 (was ~0.06 before the fix)")

for dataset in ["agg_full_moments", "agg_means"]:
    tax = load_theme_assignment(dataset, THEMES_DIR)
    data = load_split("Split_A", dataset, SPLITS_DIR)
    fcols = data["feature_cols"]

    kan = SparseKAN.from_taxonomy(tax, fcols, grid_size=14, spline_order=3,
                                  grid_range=[-5.5, 5.5])
    mlp = SparseMLP.from_taxonomy(tax, fcols)

    verify_init(kan, dataset, "SparseKAN")
    verify_init(mlp, dataset, "SparseMLP")

    del kan, mlp

Mounted at /content/drive
  [SparseKAN / agg_full_moments]  n=145  mean_ratio=0.852  median=0.902  corr(fan_in, ratio)=0.279
    -> expect mean/median near 1.0 (was ~0.06 before the fix)
  [SparseMLP / agg_full_moments]  n=145  mean_ratio=0.875  median=0.914  corr(fan_in, ratio)=0.241
    -> expect mean/median near 1.0 (was ~0.06 before the fix)
  [SparseKAN / agg_means]  n=50  mean_ratio=0.865  median=0.917  corr(fan_in, ratio)=0.118
    -> expect mean/median near 1.0 (was ~0.06 before the fix)
  [SparseMLP / agg_means]  n=50  mean_ratio=0.869  median=0.923  corr(fan_in, ratio)=0.252
    -> expect mean/median near 1.0 (was ~0.06 before the fix)


In [4]:
import torch
import numpy as np
from data_utils import load_split, load_theme_assignment
from pathlib import Path
from sparse_kan import SparseKAN
from sparse_mlp import SparseMLP

THEMES_DIR = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/05_themes")
SPLITS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/04_splits")


def verify_init_full(model, dataset_name, model_name):
    is_kan = hasattr(model.layer0, "base_weight")
    tensors = ([("base_weight",   model.layer0.base_weight.data),
                ("spline_scaler", model.layer0.spline_scaler.data)]
               if is_kan else
               [("weight",        model.layer0.weight.data)])

    print(f"  [{model_name} / {dataset_name}]  resolved as "
          f"{'KAN' if is_kan else 'MLP'}: {[t[0] for t in tensors]}")

    fan_ins = []
    for q in range(model.layer0.mask.shape[0]):
        n = int(model.layer0.mask[q].sum().item())
        if n >= 5:
            fan_ins.append(n)
    fan_ins = np.array(fan_ins)

    for tname, W in tensors:
        ratios = []
        for q in range(model.layer0.mask.shape[0]):
            active = model.layer0.mask[q].bool()
            n = int(active.sum().item())
            if n < 5:
                continue
            ratios.append(W[q][active].abs().max().item() * n**0.5)
        ratios = np.array(ratios)
        adj = ratios / (fan_ins / (fan_ins + 1))
        print(f"    {tname:<14} raw mean={ratios.mean():.3f} med={np.median(ratios):.3f}"
              f"  |  order-stat adjusted mean={adj.mean():.3f} med={np.median(adj):.3f}")

    print(f"    masking: ", end="")
    model.verify_masking()


def check_coupling(model, x0, label):
    with torch.no_grad():
        x1 = model.layer0(x0)
    stds   = x1.std(0).numpy()
    fan_in = model.layer0.mask.sum(1).numpy()
    keep   = fan_in > 0
    corr   = np.corrcoef(fan_in[keep], stds[keep])[0, 1]
    print(f"  [{label}] corr(fan_in, pre-BN std) = {corr:.3f}   (was 0.539)")
    print(f"    pre-BN std: min={stds[keep].min():.4f} "
          f"med={np.median(stds[keep]):.4f} max={stds[keep].max():.4f}  "
          f"spread={stds[keep].max()/stds[keep].min():.0f}x")


# ═══════════════════════════════════════════════════════════════════════════
# RUN CHECK A AND CHECK B FOR BOTH MODELS, BOTH DATASETS
# ═══════════════════════════════════════════════════════════════════════════

for dataset in ["agg_full_moments", "agg_means"]:
    print("=" * 80)
    print(f"  DATASET: {dataset}")
    print("=" * 80)

    tax = load_theme_assignment(dataset, THEMES_DIR)
    data = load_split("Split_A", dataset, SPLITS_DIR)
    fcols = data["feature_cols"]
    x0 = torch.tensor(data["X_train"][:2048], dtype=torch.float32)

    kan = SparseKAN.from_taxonomy(tax, fcols, grid_size=14, spline_order=3,
                                  grid_range=[-5.5, 5.5])
    mlp = SparseMLP.from_taxonomy(tax, fcols)

    print("\n--- CHECK A: init scale (base_weight + spline_scaler / weight) ---")
    verify_init_full(kan, dataset, "SparseKAN")
    verify_init_full(mlp, dataset, "SparseMLP")

    print("\n--- CHECK B: fan-in / scale coupling ---")
    check_coupling(kan, x0, f"SparseKAN / {dataset}")
    check_coupling(mlp, x0, f"SparseMLP / {dataset}")

    del kan, mlp, x0
    print()

  DATASET: agg_full_moments

--- CHECK A: init scale (base_weight + spline_scaler / weight) ---
  [SparseKAN / agg_full_moments]  resolved as KAN: ['base_weight', 'spline_scaler']
    base_weight    raw mean=0.876 med=0.908  |  order-stat adjusted mean=1.003 med=1.031
    spline_scaler  raw mean=0.875 med=0.907  |  order-stat adjusted mean=1.001 med=1.026
    masking:   ✓ All masked parameters are exactly zero and finite
  [SparseMLP / agg_full_moments]  resolved as MLP: ['weight']
    weight         raw mean=0.869 med=0.913  |  order-stat adjusted mean=0.994 med=1.031
    masking:   ✓ All masked parameters are exactly zero

--- CHECK B: fan-in / scale coupling ---
  [SparseKAN / agg_full_moments] corr(fan_in, pre-BN std) = 0.059   (was 0.539)
    pre-BN std: min=0.0089 med=0.3512 max=1.1836  spread=132x
  [SparseMLP / agg_full_moments] corr(fan_in, pre-BN std) = 0.009   (was 0.539)
    pre-BN std: min=0.0537 med=0.5245 max=1.5115  spread=28x

  DATASET: agg_means

--- CHECK A: init sc

In [7]:
import torch
import numpy as np
from data_utils import load_split, load_theme_assignment
from pathlib import Path
from sparse_kan import SparseKAN

THEMES_DIR = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/05_themes")
SPLITS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/04_splits")

# Wider grid, including values below 1e-5 since your min pre-BN std (0.0089)
# implies variance ~7.9e-5 -- worth checking whether 1e-5 alone is enough or
# whether you need to go lower still.
EPS_CANDIDATES = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7]


def sweep(z, name):
    print(f"\n  {name}  (n_units={z.shape[1]})")
    for eps in EPS_CANDIDATES:
        y = (z - z.mean(0)) / torch.sqrt(z.var(0, unbiased=False) + eps)
        s = y.std(0)
        per_unit_clamp = (y.abs() > 5).float().mean(0)
        print(f"    eps={eps:<8} unit-std min={s.min():.3f} med={s.median():.3f} "
              f"| max|x|={y.abs().max():.1f} clamp={per_unit_clamp.mean():.3%} "
              f"units>1%={int((per_unit_clamp > 0.01).sum())}")


for dataset in ["agg_full_moments", "agg_means"]:
    print("=" * 80)
    print(f"  DATASET: {dataset}")
    print("=" * 80)

    tax = load_theme_assignment(dataset, THEMES_DIR)
    data = load_split("Split_A", dataset, SPLITS_DIR)
    fcols = data["feature_cols"]

    model = SparseKAN.from_taxonomy(tax, fcols, grid_size=14, spline_order=3,
                                    grid_range=[-5.5, 5.5])
    x0 = torch.tensor(data["X_train"][:2048], dtype=torch.float32)

    with torch.no_grad():
        x1 = model.layer0(x0)

    sweep(x1, "bn1 / subtheme scores")

    with torch.no_grad():
        # bn2 provisional pass -- uses 1e-5 as a placeholder for h1's own BN,
        # just to get a realistic x2 sample. Re-run with your CHOSEN eps
        # for h1 once you've picked one (see block below).
        h1 = torch.clamp(
            (x1 - x1.mean(0)) / torch.sqrt(x1.var(0, unbiased=False) + 1e-5),
            -5, 5
        )
        x2 = model.layer1(h1)

    sweep(x2, "bn2 / theme scores (provisional -- h1 used placeholder eps=1e-5)")

    del model, x0, x1, h1, x2
    print()

  DATASET: agg_full_moments

  bn1 / subtheme scores  (n_units=331)
    eps=0.1      unit-std min=0.009 med=0.760 | max|x|=10.3 clamp=0.052% units>1%=4
    eps=0.01     unit-std min=0.029 med=0.965 | max|x|=14.5 clamp=0.227% units>1%=17
    eps=0.001    unit-std min=0.093 med=0.997 | max|x|=16.5 clamp=0.285% units>1%=23
    eps=0.0001   unit-std min=0.282 med=1.000 | max|x|=17.7 clamp=0.300% units>1%=25
    eps=1e-05    unit-std min=0.681 med=1.000 | max|x|=17.8 clamp=0.302% units>1%=26
    eps=1e-06    unit-std min=0.947 med=1.000 | max|x|=17.8 clamp=0.302% units>1%=26
    eps=1e-07    unit-std min=0.995 med=1.000 | max|x|=17.8 clamp=0.302% units>1%=26

  bn2 / theme scores (provisional -- h1 used placeholder eps=1e-5)  (n_units=13)
    eps=0.1      unit-std min=0.551 med=0.700 | max|x|=5.7 clamp=0.079% units>1%=1
    eps=0.01     unit-std min=0.902 med=0.952 | max|x|=6.5 clamp=0.192% units>1%=1
    eps=0.001    unit-std min=0.989 med=0.995 | max|x|=6.6 clamp=0.203% units>1%=1
    eps

In [9]:
CHOSEN_EPS = 1e-5  # <-- set this from the bn1 sweep above, e.g. 1e-5

assert CHOSEN_EPS is not None, "Set CHOSEN_EPS before running this block"

for dataset in ["agg_full_moments", "agg_means"]:
    print("=" * 80)
    print(f"  DATASET: {dataset}  -- bn2 re-check with CHOSEN_EPS={CHOSEN_EPS}")
    print("=" * 80)

    tax = load_theme_assignment(dataset, THEMES_DIR)
    data = load_split("Split_A", dataset, SPLITS_DIR)
    fcols = data["feature_cols"]

    model = SparseKAN.from_taxonomy(tax, fcols, grid_size=14, spline_order=3,
                                    grid_range=[-5.5, 5.5])
    x0 = torch.tensor(data["X_train"][:2048], dtype=torch.float32)

    with torch.no_grad():
        x1 = model.layer0(x0)
        h1 = torch.clamp(
            (x1 - x1.mean(0)) / torch.sqrt(x1.var(0, unbiased=False) + CHOSEN_EPS),
            -5, 5
        )
        x2 = model.layer1(h1)

    sweep(x2, f"bn2 / theme scores (h1 uses chosen eps={CHOSEN_EPS})")

    del model, x0, x1, h1, x2
    print()

  DATASET: agg_full_moments  -- bn2 re-check with CHOSEN_EPS=1e-05

  bn2 / theme scores (h1 uses chosen eps=1e-05)  (n_units=13)
    eps=0.1      unit-std min=0.580 med=0.712 | max|x|=5.1 clamp=0.008% units>1%=0
    eps=0.01     unit-std min=0.914 med=0.955 | max|x|=6.8 clamp=0.150% units>1%=1
    eps=0.001    unit-std min=0.991 med=0.995 | max|x|=7.1 clamp=0.229% units>1%=1
    eps=0.0001   unit-std min=0.999 med=1.000 | max|x|=7.1 clamp=0.229% units>1%=1
    eps=1e-05    unit-std min=1.000 med=1.000 | max|x|=7.1 clamp=0.229% units>1%=1
    eps=1e-06    unit-std min=1.000 med=1.000 | max|x|=7.1 clamp=0.229% units>1%=1
    eps=1e-07    unit-std min=1.000 med=1.000 | max|x|=7.1 clamp=0.229% units>1%=1

  DATASET: agg_means  -- bn2 re-check with CHOSEN_EPS=1e-05

  bn2 / theme scores (h1 uses chosen eps=1e-05)  (n_units=13)
    eps=0.1      unit-std min=0.529 med=0.712 | max|x|=6.8 clamp=0.004% units>1%=0
    eps=0.01     unit-std min=0.892 med=0.955 | max|x|=8.0 clamp=0.154% units>1%=1

In [10]:
tax = load_theme_assignment("agg_full_moments", THEMES_DIR)
data = load_split("Split_A", "agg_full_moments", SPLITS_DIR)
fcols = data["feature_cols"]
model = SparseKAN.from_taxonomy(tax, fcols, grid_size=14, spline_order=3,
                                grid_range=[-5.5, 5.5])
x0 = torch.tensor(data["X_train"][:2048], dtype=torch.float32)

with torch.no_grad():
    x1 = model.layer0(x0)

y = (x1 - x1.mean(0)) / torch.sqrt(x1.var(0, unbiased=False) + 1e-5)
s = y.std(0)
print(f"units below 0.9: {int((s < 0.9).sum())} / {len(s)}")

units below 0.9: 0 / 331


In [11]:
import torch
import random
import numpy as np
from data_utils import load_split, load_theme_assignment
from pathlib import Path
from sparse_kan import SparseKAN

THEMES_DIR = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/05_themes")
SPLITS_DIR = Path("/content/drive/MyDrive/Thesis/Data/Stage_5_Model_Ready/04_splits")

CHOSEN_EPS = 1e-5
SEEDS = [42, 123, 456]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


for dataset in ["agg_full_moments", "agg_means"]:
    print("=" * 80)
    print(f"  DATASET: {dataset}")
    print("=" * 80)

    tax = load_theme_assignment(dataset, THEMES_DIR)
    data = load_split("Split_A", dataset, SPLITS_DIR)
    fcols = data["feature_cols"]
    x0 = torch.tensor(data["X_train"][:2048], dtype=torch.float32)

    for seed in SEEDS:
        set_seed(seed)

        model = SparseKAN.from_taxonomy(tax, fcols, grid_size=14, spline_order=3,
                                        grid_range=[-5.5, 5.5])

        with torch.no_grad():
            x1 = model.layer0(x0)

        y = (x1 - x1.mean(0)) / torch.sqrt(x1.var(0, unbiased=False) + CHOSEN_EPS)
        s = y.std(0)

        n_below_09 = int((s < 0.9).sum().item())
        n_below_06 = int((s < 0.6).sum().item())

        print(f"  seed={seed:<5} bn1: min_unit_std={s.min().item():.3f}  "
              f"med={s.median().item():.3f}  "
              f"units<0.9={n_below_09}/{len(s)}  units<0.6={n_below_06}/{len(s)}  "
              f"max|x|={y.abs().max().item():.1f}")

        del model, x1, y, s

    print()

  DATASET: agg_full_moments
  seed=42    bn1: min_unit_std=0.932  med=1.000  units<0.9=0/331  units<0.6=0/331  max|x|=16.8
  seed=123   bn1: min_unit_std=0.979  med=1.000  units<0.9=0/331  units<0.6=0/331  max|x|=15.1
  seed=456   bn1: min_unit_std=0.902  med=1.000  units<0.9=0/331  units<0.6=0/331  max|x|=18.7

  DATASET: agg_means
  seed=42    bn1: min_unit_std=0.990  med=1.000  units<0.9=0/128  units<0.6=0/128  max|x|=13.9
  seed=123   bn1: min_unit_std=0.943  med=1.000  units<0.9=0/128  units<0.6=0/128  max|x|=12.7
  seed=456   bn1: min_unit_std=0.992  med=1.000  units<0.9=0/128  units<0.6=0/128  max|x|=12.0

